# MVSR – Vorbereitung Einheit 8
## Deep Learning: Von Gradient Descent zu CNNs, Attention und multimodalen Modellen

**C++17 / MyBinder**  
**Bearbeitungszeit:** ca. 45–60 Minuten

Dieses Notebook behandelt die Inhalte der Deep Learning Vorlesung und ist auf eine **C++-/OpenCV-Binder-Umgebung** zugeschnitten.

**Roter Faden:**

**Loss → Gradient Descent → MLP → Overfitting / Domain Shift → CNN → Attention / ViT → multimodale Embeddings**

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- Gradient Descent direkt in C++ nachvollziehen,
- den Einfluss der Lernrate untersuchen,
- erklären, warum MLPs nichtlineare Aktivierungen benötigen,
- Overfitting anhand von Trainings- und Validierungsfehler erkennen,
- ein MLP mit OpenCV auf Bilddaten trainieren,
- Domain Shift an realen Bilddaten untersuchen,
- Faltung und lokale rezeptive Felder als Grundlage von CNNs erklären,
- Attention als globale Verknüpfung von Tokens nachvollziehen,
- die Patch-Tokenisierung eines Vision Transformers erklären,
- die Grundidee gemeinsamer Bild-/Text-Embedding-Räume einordnen.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook ist für **MyBinder mit C++17 und OpenCV** vorbereitet.

Arbeiten Sie es möglichst **von oben nach unten** durch. Bereiche mit `TODO` können direkt verändert werden.

Im Unterschied zur Colab-Version wird bewusst **kein LibTorch / PyTorch-C++** vorausgesetzt. Dadurch bleibt die Binder-Umgebung klein und robust:

- Gradient Descent und ein kleines MLP werden direkt in C++ implementiert.
- Für Bildklassifikation wird `cv::ml::ANN_MLP` verwendet.
- CNN-Grundlagen werden über echte Faltungen und Feature Maps demonstriert.
- Attention wird direkt mit Matrizen berechnet.
- CLIP/VLA werden als Embedding- bzw. Architekturprinzip eingeordnet, ohne ein großes Modell herunterzuladen.

Die aktuelle Binder-Umgebung nutzt xeus-cpp; das Setup unterstützt zusätzlich das frühere xeus-cling-Layout.

Falls nach mehrmaligem Ändern von Funktionszellen eine Meldung wegen einer bereits vorhandenen Definition erscheint, starten Sie den Kernel neu und führen Sie das Notebook erneut von oben aus.

## 0. Setup

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <algorithm>
#include <numeric>
#include <random>
#include <limits>
#include <cstdlib>
#include <sstream>

#if __has_include("CppInterOp/CppInterOp.h")

    // Neueres xeus-cpp / clang-repl Setup
    #include "CppInterOp/CppInterOp.h"

    Cpp::LoadLibrary("libopencv_core.so");
    Cpp::LoadLibrary("libopencv_imgproc.so");
    Cpp::LoadLibrary("libopencv_imgcodecs.so");
    Cpp::LoadLibrary("libopencv_ml.so");

    #define MVSR_XEUS_CPP 1

#else

    // Bisheriges xeus-cling Setup
    #pragma cling add_include_path("/usr/include/opencv4")
    #pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

    #pragma cling load("opencv_core")
    #pragma cling load("opencv_imgproc")
    #pragma cling load("opencv_imgcodecs")
    #pragma cling load("opencv_ml")

#endif

#include <opencv2/opencv.hpp>
#include <opencv2/ml.hpp>

std::cout << "OpenCV-Version: "
          << CV_VERSION
          << std::endl;

std::cout << "C++ Standard: "
          << __cplusplus
          << std::endl;

### Hilfsfunktion zur Bilddarstellung

In [ ]:
#include "nlohmann/json.hpp"

#ifdef MVSR_XEUS_CPP

    #include "xeus/xbase64.hpp"
    #include "xcpp/xdisplay.hpp"

#else

    #include "xtl/xbase64.hpp"

#endif

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(
            const cv::Mat& image
        )
        {
            std::vector<unsigned char> buffer;

            cv::imencode(
                ".png",
                image,
                buffer
            );

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(
        const NotebookImage& image
    )
    {
        auto bundle = nl::json::object();

        #ifdef MVSR_XEUS_CPP

            bundle["image/png"] = xeus::base64encode(
                image.png_data
            );

        #else

            bundle["image/png"] = xtl::base64encode(
                image.png_data
            );

        #endif

        return bundle;
    }
}

#ifdef MVSR_XEUS_CPP

void show_image(
    const cv::Mat& image
)
{
    xcpp::display(
        mvsr::NotebookImage(image)
    );
}

#else

mvsr::NotebookImage show_image(
    const cv::Mat& image
)
{
    return mvsr::NotebookImage(image);
}

#endif

### Allgemeine Plot-Hilfsfunktion

In [ ]:
cv::Mat render_xy_plot(
    const std::vector<cv::Point2d>& points,
    const std::vector<cv::Point2d>& line_points,
    const std::vector<std::pair<cv::Point2d, cv::Point2d>>& segments,
    const std::string& title,
    const std::string& x_label,
    const std::string& y_label
)
{
    const int width = 760;
    const int height = 520;
    const int left = 80;
    const int right = 30;
    const int top = 45;
    const int bottom = 70;

    std::vector<cv::Point2d> all_points = points;

    all_points.insert(
        all_points.end(),
        line_points.begin(),
        line_points.end()
    );

    for (const auto& segment : segments)
    {
        all_points.push_back(segment.first);
        all_points.push_back(segment.second);
    }

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    if (all_points.empty())
    {
        return canvas;
    }

    double min_x = all_points.front().x;
    double max_x = all_points.front().x;
    double min_y = all_points.front().y;
    double max_y = all_points.front().y;

    for (const auto& p : all_points)
    {
        min_x = std::min(min_x, p.x);
        max_x = std::max(max_x, p.x);
        min_y = std::min(min_y, p.y);
        max_y = std::max(max_y, p.y);
    }

    double range_x = std::max(max_x - min_x, 1e-9);
    double range_y = std::max(max_y - min_y, 1e-9);

    min_x -= 0.08 * range_x;
    max_x += 0.08 * range_x;
    min_y -= 0.10 * range_y;
    max_y += 0.10 * range_y;

    range_x = max_x - min_x;
    range_y = max_y - min_y;

    auto to_pixel =
        [&](const cv::Point2d& p)
        {
            int px = cvRound(
                left
                + (p.x - min_x) / range_x
                * (width - left - right)
            );

            int py = cvRound(
                height - bottom
                - (p.y - min_y) / range_y
                * (height - top - bottom)
            );

            return cv::Point(px, py);
        };

    cv::line(
        canvas,
        cv::Point(left, top),
        cv::Point(left, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    cv::line(
        canvas,
        cv::Point(left, height - bottom),
        cv::Point(width - right, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    for (const auto& segment : segments)
    {
        cv::line(
            canvas,
            to_pixel(segment.first),
            to_pixel(segment.second),
            cv::Scalar(180, 180, 180),
            1,
            cv::LINE_AA
        );
    }

    for (std::size_t i = 1; i < line_points.size(); ++i)
    {
        cv::line(
            canvas,
            to_pixel(line_points[i - 1]),
            to_pixel(line_points[i]),
            cv::Scalar(40, 80, 220),
            2,
            cv::LINE_AA
        );
    }

    for (const auto& p : points)
    {
        cv::circle(
            canvas,
            to_pixel(p),
            4,
            cv::Scalar(20, 140, 30),
            -1,
            cv::LINE_AA
        );
    }

    cv::putText(
        canvas,
        title,
        cv::Point(20, 30),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(0, 0, 0),
        2
    );

    cv::putText(
        canvas,
        x_label,
        cv::Point(width / 2 - 70, height - 20),
        cv::FONT_HERSHEY_SIMPLEX,
        0.5,
        cv::Scalar(0, 0, 0),
        1
    );

    cv::putText(
        canvas,
        y_label,
        cv::Point(8, 50),
        cv::FONT_HERSHEY_SIMPLEX,
        0.45,
        cv::Scalar(0, 0, 0),
        1
    );

    return canvas;
}

### Plot für mehrere Kurven

In [ ]:
cv::Mat render_xy_series_plot(
    const std::vector<std::vector<cv::Point2d>>& series,
    const std::vector<std::string>& labels,
    const std::string& title,
    const std::string& x_label,
    const std::string& y_label
)
{
    const int width = 820;
    const int height = 520;
    const int left = 80;
    const int right = 30;
    const int top = 50;
    const int bottom = 70;

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    bool first = true;
    double min_x = 0.0;
    double max_x = 1.0;
    double min_y = 0.0;
    double max_y = 1.0;

    for (const auto& curve : series)
    {
        for (const auto& p : curve)
        {
            if (first)
            {
                min_x = max_x = p.x;
                min_y = max_y = p.y;
                first = false;
            }
            else
            {
                min_x = std::min(min_x, p.x);
                max_x = std::max(max_x, p.x);
                min_y = std::min(min_y, p.y);
                max_y = std::max(max_y, p.y);
            }
        }
    }

    if (first)
    {
        return canvas;
    }

    double range_x = std::max(max_x - min_x, 1e-12);
    double range_y = std::max(max_y - min_y, 1e-12);

    min_x -= 0.05 * range_x;
    max_x += 0.05 * range_x;
    min_y -= 0.08 * range_y;
    max_y += 0.08 * range_y;

    range_x = max_x - min_x;
    range_y = max_y - min_y;

    auto to_pixel =
        [&](const cv::Point2d& p)
        {
            int px = cvRound(
                left
                + (p.x - min_x) / range_x
                * (width - left - right)
            );

            int py = cvRound(
                height - bottom
                - (p.y - min_y) / range_y
                * (height - top - bottom)
            );

            return cv::Point(px, py);
        };

    cv::line(
        canvas,
        cv::Point(left, top),
        cv::Point(left, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    cv::line(
        canvas,
        cv::Point(left, height - bottom),
        cv::Point(width - right, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    std::vector<cv::Scalar> colors = {
        cv::Scalar(220, 70, 40),
        cv::Scalar(50, 150, 40),
        cv::Scalar(40, 80, 220),
        cv::Scalar(170, 80, 170)
    };

    for (std::size_t curve_index = 0;
         curve_index < series.size();
         ++curve_index)
    {
        const auto& curve = series[curve_index];
        cv::Scalar color = colors[curve_index % colors.size()];

        for (std::size_t i = 1; i < curve.size(); ++i)
        {
            cv::line(
                canvas,
                to_pixel(curve[i - 1]),
                to_pixel(curve[i]),
                color,
                2,
                cv::LINE_AA
            );
        }
    }

    cv::putText(
        canvas,
        title,
        cv::Point(20, 30),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(0, 0, 0),
        2
    );

    cv::putText(
        canvas,
        x_label,
        cv::Point(width / 2 - 70, height - 20),
        cv::FONT_HERSHEY_SIMPLEX,
        0.5,
        cv::Scalar(0, 0, 0),
        1
    );

    cv::putText(
        canvas,
        y_label,
        cv::Point(8, 50),
        cv::FONT_HERSHEY_SIMPLEX,
        0.45,
        cv::Scalar(0, 0, 0),
        1
    );

    for (std::size_t i = 0; i < labels.size() && i < series.size(); ++i)
    {
        cv::Scalar color = colors[i % colors.size()];
        int y = 48 + static_cast<int>(22 * i);

        cv::line(
            canvas,
            cv::Point(width - 220, y),
            cv::Point(width - 190, y),
            color,
            3
        );

        cv::putText(
            canvas,
            labels[i],
            cv::Point(width - 180, y + 5),
            cv::FONT_HERSHEY_SIMPLEX,
            0.42,
            cv::Scalar(0, 0, 0),
            1
        );
    }

    return canvas;
}

### Hilfsfunktion für Downloads

In [ ]:
bool download_file(
    const std::string& url,
    const std::string& output_file
)
{
    std::string command =
        "wget -q \"" + url + "\" -O \"" + output_file + "\"";

    int result = std::system(command.c_str());

    if (result != 0)
    {
        std::cerr << "Download fehlgeschlagen: "
                  << url << std::endl;

        return false;
    }

    return true;
}

## 1. Lernen als Funktionsapproximation

Beim überwachten Lernen wird eine Abbildung aus Beispielen angenähert:

$X \rightarrow y$

Das Modell besitzt Parameter $\theta$, die so angepasst werden, dass

$f_{\theta}(X) \approx y$

gilt.

Wir beginnen mit einem linearen Modell:

$\hat y = kx+d$

In [ ]:
std::mt19937 rng_data(42);
std::normal_distribution<double> noise_distribution(0.0, 0.6);

std::vector<double> x_data;
std::vector<double> y_data;
std::vector<cv::Point2d> regression_points;

const double true_k = 2.5;
const double true_d = -0.7;

for (int i = 0; i < 80; ++i)
{
    double x = -2.0 + 4.0 * i / 79.0;
    double y = true_k * x + true_d + noise_distribution(rng_data);

    x_data.push_back(x);
    y_data.push_back(y);
    regression_points.emplace_back(x, y);
}

std::vector<cv::Point2d> empty_line_01;
std::vector<std::pair<cv::Point2d, cv::Point2d>> empty_segments_01;

cv::Mat regression_data_plot = render_xy_plot(
    regression_points,
    empty_line_01,
    empty_segments_01,
    "Trainingsdaten",
    "x",
    "y"
);

show_image(regression_data_plot)

### Fehlerfunktion / Loss

Wir verwenden den Mean Squared Error:

$L(k,d)=\frac{1}{N}\sum_i(\hat y_i-y_i)^2$

Der Loss beschreibt, wie gut ein Parametersatz die Trainingsdaten erklärt.

In [ ]:
double linear_mse(
    const std::vector<double>& x,
    const std::vector<double>& y,
    double k,
    double d
)
{
    double loss = 0.0;

    for (std::size_t i = 0; i < x.size(); ++i)
    {
        double prediction = k * x[i] + d;
        double error = prediction - y[i];
        loss += error * error;
    }

    return loss / x.size();
}

In [ ]:
std::vector<cv::Point2d> loss_vs_k;

for (int i = 0; i < 150; ++i)
{
    double k = 5.0 * i / 149.0;
    double loss = linear_mse(
        x_data,
        y_data,
        k,
        true_d
    );

    loss_vs_k.emplace_back(k, loss);
}

std::vector<std::vector<cv::Point2d>> loss_k_series = {
    loss_vs_k
};

std::vector<std::string> loss_k_labels = {
    "Loss"
};

cv::Mat loss_k_plot = render_xy_series_plot(
    loss_k_series,
    loss_k_labels,
    "Loss abhaengig von der Steigung k",
    "k",
    "MSE"
);

show_image(loss_k_plot)

## 2. Gradient Descent

Gradient Descent aktualisiert die Modellparameter iterativ:

$\theta_{t+1}=\theta_t-\eta\nabla_{\theta}L(\theta_t)$

Die Lernrate $\eta$ bestimmt die Schrittweite der Optimierung.

In [ ]:
struct LinearGDResult
{
    double k;
    double d;
    std::vector<double> loss;
};

In [ ]:
LinearGDResult train_linear_gradient_descent(
    const std::vector<double>& x,
    const std::vector<double>& y,
    double learning_rate,
    int steps,
    double k_start,
    double d_start
)
{
    LinearGDResult result;
    result.k = k_start;
    result.d = d_start;

    for (int step = 0; step < steps; ++step)
    {
        double grad_k = 0.0;
        double grad_d = 0.0;
        double loss = 0.0;

        for (std::size_t i = 0; i < x.size(); ++i)
        {
            double prediction = result.k * x[i] + result.d;
            double error = prediction - y[i];

            loss += error * error;
            grad_k += error * x[i];
            grad_d += error;
        }

        loss /= x.size();
        grad_k *= 2.0 / x.size();
        grad_d *= 2.0 / x.size();

        result.loss.push_back(loss);

        result.k -= learning_rate * grad_k;
        result.d -= learning_rate * grad_d;
    }

    return result;
}

In [ ]:
LinearGDResult linear_fit = train_linear_gradient_descent(
    x_data,
    y_data,
    0.05,
    80,
    -1.0,
    1.5
);

std::cout << "Gelernte Steigung k: "
          << linear_fit.k << std::endl;
std::cout << "Gelernter Bias d:    "
          << linear_fit.d << std::endl;

std::vector<cv::Point2d> fitted_line;

for (int i = 0; i < 100; ++i)
{
    double x = -2.0 + 4.0 * i / 99.0;
    fitted_line.emplace_back(
        x,
        linear_fit.k * x + linear_fit.d
    );
}

cv::Mat linear_fit_plot = render_xy_plot(
    regression_points,
    fitted_line,
    empty_segments_01,
    "Lineare Regression mit Gradient Descent",
    "x",
    "y"
);

show_image(linear_fit_plot)

### Mini-Aufgabe: Einfluss der Lernrate

Testen Sie unterschiedliche Werte von $\eta$.

- zu klein → langsame Konvergenz,
- passend → schneller stabiler Abstieg,
- zu groß → Oszillation oder Divergenz.

In [ ]:
std::vector<double> learning_rates = {
    0.005,
    0.05,
    0.5
};

std::vector<std::vector<cv::Point2d>> learning_rate_curves;
std::vector<std::string> learning_rate_labels;

for (double learning_rate : learning_rates)
{
    LinearGDResult result = train_linear_gradient_descent(
        x_data,
        y_data,
        learning_rate,
        60,
        -1.0,
        1.5
    );

    std::vector<cv::Point2d> curve;

    for (std::size_t i = 0; i < result.loss.size(); ++i)
    {
        curve.emplace_back(
            static_cast<double>(i),
            result.loss[i]
        );
    }

    learning_rate_curves.push_back(curve);

    std::ostringstream label;
    label << "eta = " << learning_rate;
    learning_rate_labels.push_back(label.str());
}

cv::Mat learning_rate_plot = render_xy_series_plot(
    learning_rate_curves,
    learning_rate_labels,
    "Einfluss der Lernrate",
    "Optimierungsschritt",
    "Loss"
);

show_image(learning_rate_plot)

### TODO

Ergänzen Sie beispielsweise `learning_rate = 1.0`.

Was passiert mit dem Loss? Warum ist eine große Lernrate nicht automatisch besser?

## 3. Multi-Layer Perceptron und Nichtlinearität

Eine lineare Schicht berechnet

$h=Wx+b$.

Mehrere lineare Schichten hintereinander bleiben insgesamt linear.

Erst eine nichtlineare Aktivierung

$h=\sigma(Wx+b)$

erlaubt die Approximation nichtlinearer Zusammenhänge.

Wir implementieren dafür ein kleines MLP mit **einer versteckten Schicht** direkt in C++.

In [ ]:
struct SimpleMLP
{
    std::vector<double> w1;
    std::vector<double> b1;
    std::vector<double> w2;
    double b2;
    bool use_activation;

    double predict(double x) const
    {
        double output = b2;

        for (std::size_t j = 0; j < w1.size(); ++j)
        {
            double z = w1[j] * x + b1[j];
            double h = use_activation ? std::tanh(z) : z;
            output += w2[j] * h;
        }

        return output;
    }
};

In [ ]:
SimpleMLP create_simple_mlp(
    int hidden_units,
    bool use_activation,
    unsigned int seed
)
{
    SimpleMLP model;
    model.use_activation = use_activation;
    model.b2 = 0.0;

    std::mt19937 rng(seed);
    std::normal_distribution<double> distribution_1(0.0, 0.5);
    std::normal_distribution<double> distribution_2(0.0, 0.3);

    for (int j = 0; j < hidden_units; ++j)
    {
        model.w1.push_back(distribution_1(rng));
        model.b1.push_back(0.0);
        model.w2.push_back(distribution_2(rng));
    }

    return model;
}

In [ ]:
std::vector<double> train_simple_mlp(
    SimpleMLP& model,
    const std::vector<double>& x,
    const std::vector<double>& y,
    int epochs,
    double learning_rate
)
{
    std::vector<double> loss_history;
    const std::size_t hidden = model.w1.size();

    for (int epoch = 0; epoch < epochs; ++epoch)
    {
        std::vector<double> grad_w1(hidden, 0.0);
        std::vector<double> grad_b1(hidden, 0.0);
        std::vector<double> grad_w2(hidden, 0.0);
        double grad_b2 = 0.0;
        double loss = 0.0;

        for (std::size_t i = 0; i < x.size(); ++i)
        {
            std::vector<double> hidden_value(hidden, 0.0);
            std::vector<double> hidden_derivative(hidden, 1.0);

            double prediction = model.b2;

            for (std::size_t j = 0; j < hidden; ++j)
            {
                double z = model.w1[j] * x[i] + model.b1[j];

                if (model.use_activation)
                {
                    hidden_value[j] = std::tanh(z);
                    hidden_derivative[j] =
                        1.0 - hidden_value[j] * hidden_value[j];
                }
                else
                {
                    hidden_value[j] = z;
                    hidden_derivative[j] = 1.0;
                }

                prediction += model.w2[j] * hidden_value[j];
            }

            double error = prediction - y[i];
            loss += error * error;

            double d_output = 2.0 * error / x.size();
            grad_b2 += d_output;

            for (std::size_t j = 0; j < hidden; ++j)
            {
                grad_w2[j] += d_output * hidden_value[j];

                double d_hidden =
                    d_output
                    * model.w2[j]
                    * hidden_derivative[j];

                grad_w1[j] += d_hidden * x[i];
                grad_b1[j] += d_hidden;
            }
        }

        loss /= x.size();
        loss_history.push_back(loss);

        for (std::size_t j = 0; j < hidden; ++j)
        {
            model.w1[j] -= learning_rate * grad_w1[j];
            model.b1[j] -= learning_rate * grad_b1[j];
            model.w2[j] -= learning_rate * grad_w2[j];
        }

        model.b2 -= learning_rate * grad_b2;
    }

    return loss_history;
}

In [ ]:
std::vector<double> x_nonlinear;
std::vector<double> y_nonlinear;
std::vector<cv::Point2d> nonlinear_target;

for (int i = 0; i < 250; ++i)
{
    double x = -3.0 + 6.0 * i / 249.0;
    double y = std::sin(x) + 0.2 * std::sin(3.0 * x);

    x_nonlinear.push_back(x);
    y_nonlinear.push_back(y);
    nonlinear_target.emplace_back(x, y);
}

SimpleMLP linear_stack = create_simple_mlp(32, false, 1);
SimpleMLP nonlinear_mlp = create_simple_mlp(32, true, 2);

std::vector<double> linear_stack_loss = train_simple_mlp(
    linear_stack,
    x_nonlinear,
    y_nonlinear,
    800,
    0.005
);

std::vector<double> nonlinear_mlp_loss = train_simple_mlp(
    nonlinear_mlp,
    x_nonlinear,
    y_nonlinear,
    800,
    0.01
);

std::vector<cv::Point2d> linear_stack_curve;
std::vector<cv::Point2d> nonlinear_mlp_curve;

for (double x : x_nonlinear)
{
    linear_stack_curve.emplace_back(
        x,
        linear_stack.predict(x)
    );

    nonlinear_mlp_curve.emplace_back(
        x,
        nonlinear_mlp.predict(x)
    );
}

std::vector<std::vector<cv::Point2d>> mlp_curves = {
    nonlinear_target,
    linear_stack_curve,
    nonlinear_mlp_curve
};

std::vector<std::string> mlp_labels = {
    "Ziel",
    "nur lineare Schichten",
    "MLP mit tanh"
};

cv::Mat mlp_plot = render_xy_series_plot(
    mlp_curves,
    mlp_labels,
    "Warum braucht ein MLP Nichtlinearitaet?",
    "x",
    "y"
);

show_image(mlp_plot)

### Beobachtung

Auch viele lineare Hidden Units ergeben zusammen wieder nur eine lineare Funktion.

Mit `tanh` kann das MLP dagegen den nichtlinearen Zusammenhang approximieren.

## 4. Overfitting und Validierung

Hohe Modellkapazität kann dazu führen, dass ein Modell nicht nur den Zusammenhang, sondern auch das **Rauschen der Trainingsdaten** lernt.

Typisches Muster:

$L_{train}\downarrow$, während $L_{val}$ nach einem Minimum wieder steigt.

In [ ]:
double mlp_mse(
    const SimpleMLP& model,
    const std::vector<double>& x,
    const std::vector<double>& y
)
{
    double loss = 0.0;

    for (std::size_t i = 0; i < x.size(); ++i)
    {
        double error = model.predict(x[i]) - y[i];
        loss += error * error;
    }

    return loss / x.size();
}

In [ ]:
std::mt19937 rng_overfit(42);
std::normal_distribution<double> overfit_noise(0.0, 0.30);

std::vector<double> x_train_overfit;
std::vector<double> y_train_overfit;
std::vector<double> x_val_overfit;
std::vector<double> y_val_overfit;

for (int i = 0; i < 18; ++i)
{
    double x = -3.0 + 6.0 * i / 17.0;
    double y = std::sin(x) + 0.2 * std::sin(3.0 * x)
               + overfit_noise(rng_overfit);

    x_train_overfit.push_back(x);
    y_train_overfit.push_back(y);
}

for (int i = 0; i < 80; ++i)
{
    double x = -2.95 + 5.9 * i / 79.0;
    double y = std::sin(x) + 0.2 * std::sin(3.0 * x)
               + overfit_noise(rng_overfit);

    x_val_overfit.push_back(x);
    y_val_overfit.push_back(y);
}

SimpleMLP overfit_model = create_simple_mlp(80, true, 7);

std::vector<cv::Point2d> overfit_train_curve;
std::vector<cv::Point2d> overfit_val_curve;

for (int block = 0; block < 60; ++block)
{
    train_simple_mlp(
        overfit_model,
        x_train_overfit,
        y_train_overfit,
        50,
        0.01
    );

    double train_loss = mlp_mse(
        overfit_model,
        x_train_overfit,
        y_train_overfit
    );

    double val_loss = mlp_mse(
        overfit_model,
        x_val_overfit,
        y_val_overfit
    );

    overfit_train_curve.emplace_back(
        50.0 * (block + 1),
        train_loss
    );

    overfit_val_curve.emplace_back(
        50.0 * (block + 1),
        val_loss
    );
}

std::vector<std::vector<cv::Point2d>> overfit_series = {
    overfit_train_curve,
    overfit_val_curve
};

std::vector<std::string> overfit_labels = {
    "Training",
    "Validierung"
};

cv::Mat overfit_plot = render_xy_series_plot(
    overfit_series,
    overfit_labels,
    "Training vs. Validierung",
    "Epoche",
    "MSE"
);

show_image(overfit_plot)

### Diskussion

- An welcher Stelle ist der Validierungsfehler minimal?
- Warum ist das Modell danach auf den Trainingsdaten noch besser, aber auf ungesehenen Daten schlechter?
- Welche Gegenmaßnahmen kennen Sie?

Typisch sind **weniger Modellkapazität, Regularisierung, mehr Daten, Data Augmentation und Early Stopping**.

## 5. Bildklassifikation mit einem MLP

Für ein Bild mit $20\times20$ Pixeln erhält ein klassisches MLP einen Vektor der Länge

$20\cdot20=400$.

Damit geht die explizite räumliche Nachbarschaft verloren: Das MLP sieht zunächst nur **400 Eingabedimensionen**.

Wir verwenden den offiziellen OpenCV-Testdatensatz `digits.png` mit 5000 handgeschriebenen Ziffern.

In [ ]:
std::system("mkdir -p dl_data");

download_file(
    "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/digits.png",
    "dl_data/digits.png"
);

cv::Mat digits_image = cv::imread(
    "dl_data/digits.png",
    cv::IMREAD_GRAYSCALE
);

std::cout << "Datensatzgroesse: "
          << digits_image.cols << " x "
          << digits_image.rows << std::endl;

In [ ]:
cv::Mat extract_digit_cell(
    const cv::Mat& digits,
    int sample_index
)
{
    const int cell_size = 20;
    const int cells_per_row = digits.cols / cell_size;

    int row = sample_index / cells_per_row;
    int col = sample_index % cells_per_row;

    return digits(
        cv::Rect(
            col * cell_size,
            row * cell_size,
            cell_size,
            cell_size
        )
    ).clone();
}

In [ ]:
cv::Mat example_digit = extract_digit_cell(
    digits_image,
    1234
);

cv::Mat example_digit_large;
cv::resize(
    example_digit,
    example_digit_large,
    cv::Size(240, 240),
    0.0,
    0.0,
    cv::INTER_NEAREST
);

show_image(example_digit_large)

### Trainings- und Testdaten erzeugen

Pro Klasse werden 400 Bilder zum Training und 100 Bilder zum Test verwendet.

In [ ]:
cv::Mat mlp_train_data(4000, 400, CV_32F);
cv::Mat mlp_train_targets = cv::Mat::zeros(4000, 10, CV_32F);
cv::Mat mlp_test_data(1000, 400, CV_32F);
cv::Mat mlp_test_labels(1000, 1, CV_32S);

std::vector<int> mlp_test_indices;

int train_row = 0;
int test_row = 0;

for (int digit = 0; digit < 10; ++digit)
{
    for (int local_index = 0; local_index < 500; ++local_index)
    {
        int sample_index = digit * 500 + local_index;
        cv::Mat cell = extract_digit_cell(digits_image, sample_index);

        cv::Mat cell_float;
        cell.convertTo(cell_float, CV_32F, 1.0 / 255.0);

        cv::Mat flattened = cell_float.reshape(1, 1);

        if (local_index < 400)
        {
            flattened.copyTo(mlp_train_data.row(train_row));
            mlp_train_targets.at<float>(train_row, digit) = 1.0F;
            ++train_row;
        }
        else
        {
            flattened.copyTo(mlp_test_data.row(test_row));
            mlp_test_labels.at<int>(test_row, 0) = digit;
            mlp_test_indices.push_back(sample_index);
            ++test_row;
        }
    }
}

std::cout << "Training: " << mlp_train_data.rows << " Bilder" << std::endl;
std::cout << "Test:     " << mlp_test_data.rows << " Bilder" << std::endl;

### OpenCV-MLP trainieren

Das Netzwerk besitzt

$400 \rightarrow 64 \rightarrow 10$

Neuronen.

In [ ]:
cv::Ptr<cv::ml::ANN_MLP> digit_mlp = cv::ml::ANN_MLP::create();

cv::Mat layer_sizes = (
    cv::Mat_<int>(1, 3)
    << 400, 64, 10
);

digit_mlp->setLayerSizes(layer_sizes);
digit_mlp->setActivationFunction(
    cv::ml::ANN_MLP::SIGMOID_SYM,
    1.0,
    1.0
);

digit_mlp->setTrainMethod(
    cv::ml::ANN_MLP::BACKPROP,
    0.001
);

digit_mlp->setTermCriteria(
    cv::TermCriteria(
        cv::TermCriteria::MAX_ITER + cv::TermCriteria::EPS,
        300,
        1e-4
    )
);

cv::Ptr<cv::ml::TrainData> digit_train_data = cv::ml::TrainData::create(
    mlp_train_data,
    cv::ml::ROW_SAMPLE,
    mlp_train_targets
);

bool training_ok = digit_mlp->train(digit_train_data);

std::cout << "Training erfolgreich: "
          << std::boolalpha << training_ok
          << std::endl;

In [ ]:
double evaluate_digit_mlp(
    const cv::Ptr<cv::ml::ANN_MLP>& model,
    const cv::Mat& samples,
    const cv::Mat& labels
)
{
    int correct = 0;

    for (int i = 0; i < samples.rows; ++i)
    {
        cv::Mat output;
        model->predict(samples.row(i), output);

        cv::Point max_location;
        cv::minMaxLoc(output, nullptr, nullptr, nullptr, &max_location);

        int predicted = max_location.x;
        int ground_truth = labels.at<int>(i, 0);

        if (predicted == ground_truth)
        {
            ++correct;
        }
    }

    return static_cast<double>(correct) / samples.rows;
}

In [ ]:
double normal_accuracy = evaluate_digit_mlp(
    digit_mlp,
    mlp_test_data,
    mlp_test_labels
);

std::cout << "MLP Test Accuracy: "
          << std::fixed << std::setprecision(3)
          << normal_accuracy
          << std::endl;

## 6. Warum CNNs für Bilder?

Beim MLP besitzt bereits die erste Schicht viele Parameter:

$400\cdot64+64=25\,664$.

Eine Convolution mit 16 Filtern der Größe $3\times3$ benötigt dagegen nur

$16\cdot(3\cdot3+1)=160$

Parameter.

Der Grund: **lokale rezeptive Felder + geteilte Gewichte**.

In [ ]:
int mlp_first_layer_parameters = 400 * 64 + 64;
int conv_layer_parameters = 16 * (3 * 3 + 1);

std::cout << "MLP erste Schicht: "
          << mlp_first_layer_parameters
          << " Parameter" << std::endl;

std::cout << "Conv-Schicht:      "
          << conv_layer_parameters
          << " Parameter" << std::endl;

### Faltung und Feature Maps

In einem trainierten CNN werden die Kernelwerte gelernt.

Hier verwenden wir bewusst bekannte Kantenfilter, um zunächst die **Operation** einer Convolution sichtbar zu machen.

In [ ]:
cv::Mat kernel_vertical = (
    cv::Mat_<float>(3, 3)
    << -1, 0, 1,
       -2, 0, 2,
       -1, 0, 1
);

cv::Mat kernel_horizontal = (
    cv::Mat_<float>(3, 3)
    << -1, -2, -1,
        0,  0,  0,
        1,  2,  1
);

cv::Mat feature_vertical;
cv::Mat feature_horizontal;

cv::filter2D(
    example_digit,
    feature_vertical,
    CV_32F,
    kernel_vertical
);

cv::filter2D(
    example_digit,
    feature_horizontal,
    CV_32F,
    kernel_horizontal
);

cv::Mat feature_vertical_abs;
cv::Mat feature_horizontal_abs;

cv::convertScaleAbs(feature_vertical, feature_vertical_abs);
cv::convertScaleAbs(feature_horizontal, feature_horizontal_abs);

cv::Mat digit_large;
cv::Mat vertical_large;
cv::Mat horizontal_large;

cv::resize(example_digit, digit_large, cv::Size(240, 240), 0, 0, cv::INTER_NEAREST);
cv::resize(feature_vertical_abs, vertical_large, cv::Size(240, 240), 0, 0, cv::INTER_NEAREST);
cv::resize(feature_horizontal_abs, horizontal_large, cv::Size(240, 240), 0, 0, cv::INTER_NEAREST);

std::vector<cv::Mat> convolution_panels = {
    digit_large,
    vertical_large,
    horizontal_large
};

cv::Mat convolution_overview;
cv::hconcat(convolution_panels, convolution_overview);

show_image(convolution_overview)

**Von links nach rechts:** Eingangsbild, Reaktion auf vertikale Kanten, Reaktion auf horizontale Kanten.

Ein CNN lernt solche Filterwerte aus Daten. Entscheidend bleibt: Ein $3\times3$-Kernel verarbeitet zunächst nur eine **lokale Nachbarschaft**.

## 7. Domain Shift

Ein Modell kann auf einem klassischen Testdatensatz gut funktionieren und trotzdem bei veränderten Eingabebedingungen stark schlechter werden.

Wir drehen die Testbilder und fügen Rauschen hinzu. Das MLP wurde auf diesen Veränderungen **nicht trainiert**.

In [ ]:
cv::Mat create_shifted_digit(
    const cv::Mat& input,
    double angle_degrees,
    double noise_sigma
)
{
    cv::Point2f center(
        input.cols / 2.0F,
        input.rows / 2.0F
    );

    cv::Mat rotation = cv::getRotationMatrix2D(
        center,
        angle_degrees,
        1.0
    );

    cv::Mat rotated;
    cv::warpAffine(
        input,
        rotated,
        rotation,
        input.size(),
        cv::INTER_LINEAR,
        cv::BORDER_CONSTANT,
        cv::Scalar(0)
    );

    cv::Mat noise(input.size(), CV_16S);
    cv::randn(noise, 0.0, noise_sigma);

    cv::Mat rotated_16;
    rotated.convertTo(rotated_16, CV_16S);

    cv::Mat noisy_16 = rotated_16 + noise;
    cv::Mat result;
    noisy_16.convertTo(result, CV_8U);

    return result;
}

In [ ]:
cv::theRNG().state = 42;

cv::Mat shifted_test_data(1000, 400, CV_32F);

for (int i = 0; i < 1000; ++i)
{
    cv::Mat original = extract_digit_cell(
        digits_image,
        mlp_test_indices[i]
    );

    cv::Mat shifted = create_shifted_digit(
        original,
        25.0,
        30.0
    );

    cv::Mat shifted_float;
    shifted.convertTo(shifted_float, CV_32F, 1.0 / 255.0);

    cv::Mat flattened = shifted_float.reshape(1, 1);
    flattened.copyTo(shifted_test_data.row(i));
}

double shifted_accuracy = evaluate_digit_mlp(
    digit_mlp,
    shifted_test_data,
    mlp_test_labels
);

std::cout << "Accuracy normal:       "
          << normal_accuracy << std::endl;

std::cout << "Accuracy Domain Shift: "
          << shifted_accuracy << std::endl;

In [ ]:
cv::Mat normal_digit_demo = extract_digit_cell(
    digits_image,
    mlp_test_indices[5]
);

cv::Mat shifted_digit_demo = create_shifted_digit(
    normal_digit_demo,
    25.0,
    30.0
);

cv::Mat normal_digit_large;
cv::Mat shifted_digit_large;

cv::resize(normal_digit_demo, normal_digit_large, cv::Size(260, 260), 0, 0, cv::INTER_NEAREST);
cv::resize(shifted_digit_demo, shifted_digit_large, cv::Size(260, 260), 0, 0, cv::INTER_NEAREST);

std::vector<cv::Mat> shift_panels = {
    normal_digit_large,
    shifted_digit_large
};

cv::Mat shift_overview;
cv::hconcat(shift_panels, shift_overview);

show_image(shift_overview)

### Interpretation

Eine gute Test-Accuracy bedeutet nicht automatisch, dass das Modell unter realen Änderungen robust ist.

Für Robotik ist deshalb entscheidend:

**Entspricht die Trainingsverteilung dem späteren Einsatzgebiet?**

## 8. Von lokalen CNN-Filtern zu Attention

CNNs verarbeiten lokale Nachbarschaften effizient.

Attention ermöglicht dagegen direkte Beziehungen zwischen beliebigen Tokens:

$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

Wir betrachten vier kleine Token-Vektoren. Token 1 und 2 sind bewusst ähnlich.

In [ ]:
cv::Mat row_softmax(
    const cv::Mat& input
)
{
    cv::Mat output(input.size(), CV_64F);

    for (int r = 0; r < input.rows; ++r)
    {
        double max_value = -std::numeric_limits<double>::infinity();

        for (int c = 0; c < input.cols; ++c)
        {
            max_value = std::max(
                max_value,
                input.at<double>(r, c)
            );
        }

        double sum = 0.0;

        for (int c = 0; c < input.cols; ++c)
        {
            double value = std::exp(
                input.at<double>(r, c) - max_value
            );

            output.at<double>(r, c) = value;
            sum += value;
        }

        for (int c = 0; c < input.cols; ++c)
        {
            output.at<double>(r, c) /= sum;
        }
    }

    return output;
}

In [ ]:
cv::Mat render_heatmap(
    const cv::Mat& matrix,
    const std::string& title
)
{
    cv::Mat normalized;
    cv::normalize(
        matrix,
        normalized,
        0,
        255,
        cv::NORM_MINMAX,
        CV_8U
    );

    cv::Mat colored;
    cv::applyColorMap(
        normalized,
        colored,
        cv::COLORMAP_VIRIDIS
    );

    cv::resize(
        colored,
        colored,
        cv::Size(420, 420),
        0,
        0,
        cv::INTER_NEAREST
    );

    cv::putText(
        colored,
        title,
        cv::Point(12, 28),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(255, 255, 255),
        2
    );

    return colored;
}

In [ ]:
cv::Mat tokens = (
    cv::Mat_<double>(4, 3)
    << 1.0, 0.0, 0.2,
       0.9, 0.1, 0.2,
       0.0, 1.0, 0.1,
       0.1, 0.0, 1.0
);

cv::Mat attention_logits =
    tokens * tokens.t()
    / std::sqrt(3.0);

cv::Mat attention_weights = row_softmax(
    attention_logits
);

std::cout << "Attention-Matrix:" << std::endl;
std::cout << attention_weights << std::endl;

cv::Mat attention_heatmap = render_heatmap(
    attention_weights,
    "Attention-Matrix"
);

show_image(attention_heatmap)

### Mini-Aufgabe

Verändern Sie den vierten Token so, dass er Token 1 sehr ähnlich wird.

Wie verändert sich die Attention-Matrix?

Die zentrale Idee ist: **Jeder Query-Token kann direkt Information von jedem Key-Token beziehen.**

## 9. Vision Transformer: Bilder als Patch-Sequenz

Ein Vision Transformer zerlegt ein Bild in nicht überlappende Patches.

Für unser $20\times20$-Bild und eine Patchgröße von $5\times5$ entstehen

$4\cdot4=16$

Patch-Tokens.

Jeder Patch wird abgeflacht und anschließend linear in einen Embedding-Raum projiziert.

In [ ]:
cv::Mat patch_visualization;
cv::resize(
    normal_digit_demo,
    patch_visualization,
    cv::Size(400, 400),
    0,
    0,
    cv::INTER_NEAREST
);

for (int i = 1; i < 4; ++i)
{
    cv::line(
        patch_visualization,
        cv::Point(i * 100, 0),
        cv::Point(i * 100, 399),
        cv::Scalar(180),
        3
    );

    cv::line(
        patch_visualization,
        cv::Point(0, i * 100),
        cv::Point(399, i * 100),
        cv::Scalar(180),
        3
    );
}

show_image(patch_visualization)

In [ ]:
const int patch_size = 5;
const int patches_x = normal_digit_demo.cols / patch_size;
const int patches_y = normal_digit_demo.rows / patch_size;
const int number_of_patches = patches_x * patches_y;
const int values_per_patch = patch_size * patch_size;

std::cout << "Patchgroesse:       "
          << patch_size << " x " << patch_size << std::endl;
std::cout << "Anzahl Patch-Tokens: "
          << number_of_patches << std::endl;
std::cout << "Werte pro Patch:     "
          << values_per_patch << std::endl;
std::cout << "Mit CLS-Token:       "
          << number_of_patches + 1
          << " Tokens" << std::endl;

### MLP vs. CNN vs. ViT

**MLP**  
Pixel → Flatten → vollständig verbundene Abbildung

**CNN**  
Bild → lokale Filter → hierarchische Feature Maps

**ViT**  
Bild → Patches → Tokens → globale Attention

Die Architekturen unterscheiden sich damit stark darin, **wie Information zwischen Bildregionen fließt**.

## 10. Gemeinsamer Bild-/Text-Embedding-Raum – CLIP-Idee

CLIP projiziert Bilder und Texte in einen gemeinsamen Embedding-Raum:

$\mathbf z_I=f_{image}(I)$

$\mathbf z_T=f_{text}(T)$

Passende Bild-/Text-Paare sollen eine hohe Ähnlichkeit besitzen.

Ein vollständiges CLIP-Modell wird in dieser schlanken C++-Binder-Umgebung bewusst **nicht** geladen. Das folgende Toy-Beispiel zeigt nur das Prinzip der Cosine Similarity.

In [ ]:
double cosine_similarity(
    const std::vector<double>& a,
    const std::vector<double>& b
)
{
    double dot = 0.0;
    double norm_a = 0.0;
    double norm_b = 0.0;

    for (std::size_t i = 0; i < a.size(); ++i)
    {
        dot += a[i] * b[i];
        norm_a += a[i] * a[i];
        norm_b += b[i] * b[i];
    }

    return dot / (
        std::sqrt(norm_a)
        * std::sqrt(norm_b)
        + 1e-12
    );
}

In [ ]:
std::vector<double> image_embedding = {
    0.85, 0.15, 0.10, 0.05
};

std::vector<std::string> text_prompts = {
    "a handwritten digit",
    "a robot arm",
    "a landscape",
    "a handwritten number"
};

std::vector<std::vector<double>> text_embeddings = {
    {0.90, 0.10, 0.05, 0.05},
    {0.10, 0.85, 0.10, 0.10},
    {0.05, 0.10, 0.90, 0.15},
    {0.82, 0.18, 0.08, 0.04}
};

for (std::size_t i = 0; i < text_prompts.size(); ++i)
{
    double similarity = cosine_similarity(
        image_embedding,
        text_embeddings[i]
    );

    std::cout << std::fixed << std::setprecision(3)
              << similarity << "  "
              << text_prompts[i]
              << std::endl;
}

**Wichtig:** Die Vektoren oben sind künstlich gewählt. Ein echtes CLIP lernt diese Repräsentationen aus sehr großen Bild-/Text-Datensätzen.

Die didaktische Kernaussage bleibt:

**Bildrepräsentation ↔ Textrepräsentationen → Ähnlichkeit → Zero-Shot-Entscheidung**

## 11. Ausblick: Vom VLM zum VLA

Ein Vision-Language-Model liefert noch keine Roboterstellgröße.

Ein VLA ergänzt eine Policy bzw. einen Action Head:

**Bild + Sprache + optional Roboterzustand → Aktion**

Sehr vereinfacht:

$(o_t,\ell)\rightarrow a_t$

Typische Trainingsbausteine:

- LLM: Self-Supervised Learning,
- Vision-/VLM-Teil: kontrastives bzw. multimodales Training,
- Action Head: Behaviour Cloning,
- optional Reinforcement Learning.

Ein vollständiges VLA wird hier bewusst nicht trainiert. Dafür wären große Modelle, Roboterdaten und erheblich mehr Rechenressourcen notwendig.

### Einordnung gegenüber Visual Servoing

**Klassisches Visual Servoing**

- expliziter visueller Fehler,
- geometrisches Modell,
- lokal präzise,
- interpretierbarer Regelkreis.

**VLA**

- Bild + Sprache + weitere Sensordaten,
- semantisch flexibler,
- Multi-Task,
- datenabhängige Policy,
- schwerer zu verifizieren.

Interessant sind deshalb auch **Hybridansätze**:

semantische Planung / Aufgabenwahl durch gelernte Modelle  
$\rightarrow$  
präzise lokale Ausführung durch klassische Regelung.

## 12. Selbstcheck

1. Was beschreibt der Loss beim Training?
2. Welche Rolle spielt die Lernrate $\eta$?
3. Warum helfen mehrere lineare Schichten alleine nicht bei nichtlinearen Problemen?
4. Woran erkennt man Overfitting?
5. Was bedeutet Domain Shift?
6. Warum ist Flattening eines Bildes für ein MLP problematisch?
7. Warum benötigt eine Convolution deutlich weniger Parameter als eine voll verbundene Schicht?
8. Was beschreibt eine Feature Map?
9. Was beschreibt eine Attention-Matrix?
10. Wie wird ein Bild für einen Vision Transformer vorbereitet?
11. Was ist die Grundidee eines gemeinsamen Bild-/Text-Embedding-Raums?
12. Warum ist ein VLM noch keine Roboterpolicy?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Er bewertet die Abweichung zwischen Modellvorhersage und Trainingsziel.  
2. Sie bestimmt die Schrittweite der Parameterupdates.  
3. Eine Verkettung linearer Abbildungen bleibt linear.  
4. Trainingsfehler sinkt weiter, während der Validierungsfehler wieder steigt.  
5. Trainings- und Einsatzverteilung unterscheiden sich.  
6. Die räumliche Nachbarschaft ist nicht explizit in der Architektur enthalten.  
7. Kernelgewichte werden lokal und über alle Bildpositionen geteilt.  
8. Die räumliche Reaktion eines Filters auf das Eingangsbild.  
9. Wie stark Tokens Information von anderen Tokens beziehen.  
10. Es wird in Patches zerlegt und in Patch-Tokens umgewandelt.  
11. Bilder und Texte werden so repräsentiert, dass semantisch passende Inhalte nahe beieinander liegen.  
12. Es erzeugt noch keine Roboteraktionen bzw. Stellgrößen.

</details>

## 13. Kleine Aufgaben

### Aufgabe A – Gradient Descent
Ergänzen Sie eine sehr große Lernrate. Ab wann divergiert das Training?

### Aufgabe B – MLP-Kapazität
Ändern Sie die Zahl der Hidden Units von `32` auf `4` bzw. `64`. Wie verändert sich die Funktionsapproximation?

### Aufgabe C – Overfitting
Reduzieren Sie die Zahl der Trainingspunkte oder erhöhen Sie die Hidden Units. Wird Overfitting deutlicher?

### Aufgabe D – Bild-MLP
Verändern Sie die Größe der Hidden-Schicht des OpenCV-MLPs. Wie verändern sich Trainingszeit und Accuracy?

### Aufgabe E – CNN
Verändern Sie die $3\times3$-Kernel. Welche Bildstrukturen werden hervorgehoben?

### Aufgabe F – Domain Shift
Verändern Sie Rotationswinkel und Rauschstärke. Wie schnell fällt die Accuracy?

### Aufgabe G – Attention
Verändern Sie die Token-Vektoren. Welche Tokens erhalten hohe gegenseitige Attention?

### Aufgabe H – ViT
Ändern Sie gedanklich die Patchgröße von $5$ auf $4$ bzw. $10$. Wie verändert sich die Tokenzahl?

## 14. Take-away

- Deep Learning approximiert komplexe funktionale Zusammenhänge durch viele lernbare Parameter.
- Gradient Descent minimiert den Loss durch iterative Parameterupdates.
- Nichtlineare Aktivierungen sind entscheidend für nichtlineare Modellierung.
- Hohe Modellkapazität erhöht auch das Risiko für Overfitting.
- Gute Testwerte schützen nicht automatisch vor Domain Shift.
- MLPs nutzen Bildgeometrie nicht explizit.
- CNNs verwenden lokale rezeptive Felder und geteilte Gewichte.
- Attention ermöglicht direkte globale Beziehungen zwischen Tokens.
- Vision Transformer repräsentieren Bilder als Patch-Token-Sequenzen.
- Multimodale Modelle verbinden Bild- und Sprachrepräsentationen.
- VLA-Modelle ergänzen multimodale Repräsentationen um eine gelernte Roboterpolicy.